# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neha-raniii/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/neha-raniii/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} rows loaded")
df.head(3)

30,000 rows loaded


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule (plain words): Flag a page for review if it is stale (not updated in 180+ days), still gets meaningful search demand (impressions_90d >= 500), AND is currently declining (trend_direction == "down"). This targets pages worth a reviewer's limited time - not just any old page, but one that's actively losing ground while still visible.

Reason codes this rule can output:
- stale_and_declining: stale + declining + visible
- ctr_below_position_norm: CTR is below what's typical for its position tier

In [8]:
# Signal 1: staleness vs decline (linked to FlyRank's real refresh flag logic)
df['is_stale'] = df['days_since_last_update'] >= 180
stale_check = df.groupby('is_stale')['trend_direction'].value_counts(normalize=True).unstack().round(3)
n_stale = df['is_stale'].value_counts()
print("Staleness vs trend direction (share within each group):")
print(stale_check)
print("\nn per group:", dict(n_stale))


Staleness vs trend direction (share within each group):
trend_direction   down   flat    new  stable     up
is_stale                                           
False            0.542  0.038  0.074   0.199  0.146
True             0.471  0.092  0.144   0.138  0.155

n per group: {False: np.int64(29826), True: np.int64(174)}


Signal 1 verdict: OPPOSITE. Stale pages (days_since_last_update >= 180, n=174) show a LOWER decline rate (47.1%) than non-stale pages (n=29,826, decline rate 54.2%). This is the opposite of what the refresh-flag intuition assumes. Caveat: the stale group is tiny (174 rows vs 29,826), so this could be noise rather than a real pattern - but it's still a clearly negative result worth reporting rather than ignoring.

In [9]:
# Signal 2: CTR vs position tier (linked to FlyRank's CTR-fix flag logic)
ctr_by_tier = df.groupby('position_tier')['ctr'].agg(['mean', 'count']).round(4)
ctr_by_tier = ctr_by_tier.sort_values('mean', ascending=False)
print("Mean CTR by position tier:")
print(ctr_by_tier)

Mean CTR by position tier:
                 mean  count
position_tier               
top_3          1.4836   2321
page_1         0.6525  11814
striking       0.3232   7304
page_3_5       0.2225   7242
deep           0.1502   1319


Signal 2 verdict: CONFIRMED. CTR drops cleanly and monotonically as position tier gets worse: top_3 (mean CTR 1.48, n=2,321) > page_1 (0.65, n=11,814) > striking (0.32, n=7,304) > page_3_5 (0.22, n=7,242). This confirms the CTR-fix logic assumption - pages with worse average position genuinely earn less clicks per impression, so CTR-vs-position is a real, usable signal for flagging under-performing pages.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Rule encoded: Flag a page if it has meaningful visibility (impressions_90d >= 500), a real position (avg_position > 0), AND its CTR is below the mean CTR for its own position tier (using the confirmed signal from Section 1). Score = how far below the tier average the page's CTR sits, scaled by impressions (so high-traffic underperformers rank first). Reason code: ctr_below_position_norm. Action: review_ctr_and_metadata.

In [10]:
import os

# Compute tier-average CTR (from confirmed Signal 2)
tier_avg_ctr = df.groupby('position_tier')['ctr'].transform('mean')

# Eligible pages: real position, real visibility
eligible = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 500)].copy()
eligible['tier_avg_ctr'] = df.loc[eligible.index, 'ctr'] if False else tier_avg_ctr.loc[eligible.index]

# Score: gap below tier average, scaled by impressions (bigger audience = bigger opportunity)
eligible['ctr_gap'] = eligible['tier_avg_ctr'] - eligible['ctr']
eligible = eligible[eligible['ctr_gap'] > 0]  # only genuine underperformers
eligible['score'] = eligible['ctr_gap'] * eligible['impressions_90d']

eligible['reason_code'] = 'ctr_below_position_norm'
eligible['action'] = 'review_ctr_and_metadata'

queue = eligible.sort_values('score', ascending=False)[
    ['content_id', 'client_id', 'position_tier', 'avg_position', 'ctr', 'tier_avg_ctr',
     'impressions_90d', 'score', 'reason_code', 'action']
].reset_index(drop=True)

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Queue written: {len(queue):,} flagged pages")
queue.head(10)


Queue written: 13,698 flagged pages


,content_id,client_id,position_tier,avg_position,ctr,tier_avg_ctr,impressions_90d,score,reason_code,action
0,content_8c19996aa890,client_4e07408562,top_3,2.5,0.15,1.483611,509252,679143.820819,ctr_below_position_norm,review_ctr_and_metadata
1,content_4c36c775b818,client_4e07408562,top_3,2.3,0.41,1.483611,463103,497192.249268,ctr_below_position_norm,review_ctr_and_metadata
2,content_8451fc6f034d,client_d029fa3a95,top_3,2.3,0.03,1.483611,272144,395591.379371,ctr_below_position_norm,review_ctr_and_metadata
3,content_5fe46e04994d,client_4e07408562,page_1,4.2,0.14,0.652467,517715,265311.627747,ctr_below_position_norm,review_ctr_and_metadata
4,content_44e481c8f55b,client_19581e27de,top_3,1.4,0.65,1.483611,312694,260665.005661,ctr_below_position_norm,review_ctr_and_metadata
5,content_e12868d1f396,client_4e07408562,top_3,2.9,0.07,1.483611,149712,211634.457079,ctr_below_position_norm,review_ctr_and_metadata
6,content_aaef01a50def,client_19581e27de,page_1,5.4,0.25,0.652467,517109,208119.083008,ctr_below_position_norm,review_ctr_and_metadata
7,content_9532f197bbc8,client_4e07408562,top_3,2.0,0.87,1.483611,309192,189723.461646,ctr_below_position_norm,review_ctr_and_metadata
8,content_4a6607efcb46,client_6208ef0f77,top_3,2.2,0.01,1.483611,128068,188722.351142,ctr_below_position_norm,review_ctr_and_metadata
9,content_36ff89c8214e,client_19581e27de,page_1,7.3,0.05,0.652467,295097,177786.075959,ctr_below_position_norm,review_ctr_and_metadata


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review: for each flagged page, the action is always review_ctr_and_metadata (rewrite title/meta or check search intent match), the reason is the CTR gap below its position tier's average. What would make each wrong: if the low CTR is actually due to a misleading title that over-promises (so fixing metadata could backfire), or if the page recently changed content/URL and CTR hasn't caught up yet, or if impressions_90d includes irrelevant/broad-match queries inflating the denominator.

In [11]:
top20 = queue.head(20).copy()
top20['confidence_note'] = top20['score'].apply(
    lambda s: 'high' if s > queue['score'].quantile(0.9) else 'medium'
)
top20[['content_id', 'position_tier', 'avg_position', 'ctr', 'tier_avg_ctr', 'impressions_90d', 'score', 'confidence_note']]


,content_id,position_tier,avg_position,ctr,tier_avg_ctr,impressions_90d,score,confidence_note
0,content_8c19996aa890,top_3,2.5,0.15,1.483611,509252,679143.820819,high
1,content_4c36c775b818,top_3,2.3,0.41,1.483611,463103,497192.249268,high
2,content_8451fc6f034d,top_3,2.3,0.03,1.483611,272144,395591.379371,high
3,content_5fe46e04994d,page_1,4.2,0.14,0.652467,517715,265311.627747,high
4,content_44e481c8f55b,top_3,1.4,0.65,1.483611,312694,260665.005661,high
5,content_e12868d1f396,top_3,2.9,0.07,1.483611,149712,211634.457079,high
6,content_aaef01a50def,page_1,5.4,0.25,0.652467,517109,208119.083008,high
7,content_9532f197bbc8,top_3,2.0,0.87,1.483611,309192,189723.461646,high
8,content_4a6607efcb46,top_3,2.2,0.01,1.483611,128068,188722.351142,high
9,content_36ff89c8214e,page_1,7.3,0.05,0.652467,295097,177786.075959,high


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Row 8 (content_4a6607efcb46) has ctr=0.01 with 128,068 impressions at top_3 position - CTR this low at such a strong position is suspicious. It could be a case of misleading title/snippet (over-promising, so people click away fast after seeing it in results without clicking), or a tracking/attribution glitch rather than a genuine content problem. Similarly row 14 (content_c8e9d6ab9013) has ctr=0.00 exactly - a true zero click rate on 208,678 impressions is unusual enough that I would sanity-check it against raw click logs before trusting the flag.

Leakage check: No product-computed flags (health_score, priority_score, action_type) were used anywhere - they are not present in this dataset. No future-window data was used - the score is built entirely from impressions_90d, avg_position, and ctr, which are all current-window observed signals, not future outcomes. trend_direction and trend_pct (label-derived fields from the earlier notebooks) were NOT used as inputs to this rule, confirming no leakage from that direction either.

In [12]:
"""Weak picks: Row 8 (content_4a6607efcb46) has ctr=0.01 with 128,068 impressions at top_3 position - CTR this low at such a strong position is suspicious. It could be a case of misleading title/snippet (over-promising, so people click away fast after seeing it in results without clicking), or a tracking/attribution glitch rather than a genuine content problem. Similarly row 14 (content_c8e9d6ab9013) has ctr=0.00 exactly - a true zero click rate on 208,678 impressions is unusual enough that I would sanity-check it against raw click logs before trusting the flag.

Leakage check: No product-computed flags (health_score, priority_score, action_type) were used anywhere - they are not present in this dataset. No future-window data was used - the score is built entirely from impressions_90d, avg_position, and ctr, which are all current-window observed signals, not future outcomes. trend_direction and trend_pct (label-derived fields from the earlier notebooks) were NOT used as inputs to this rule, confirming no leakage from that direction either."""


'Weak picks: Row 8 (content_4a6607efcb46) has ctr=0.01 with 128,068 impressions at top_3 position - CTR this low at such a strong position is suspicious. It could be a case of misleading title/snippet (over-promising, so people click away fast after seeing it in results without clicking), or a tracking/attribution glitch rather than a genuine content problem. Similarly row 14 (content_c8e9d6ab9013) has ctr=0.00 exactly - a true zero click rate on 208,678 impressions is unusual enough that I would sanity-check it against raw click logs before trusting the flag.\n\nLeakage check: No product-computed flags (health_score, priority_score, action_type) were used anywhere - they are not present in this dataset. No future-window data was used - the score is built entirely from impressions_90d, avg_position, and ctr, which are all current-window observed signals, not future outcomes. trend_direction and trend_pct (label-derived fields from the earlier notebooks) were NOT used as inputs to this 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.